# Same value, different object: `is` vs `==`
**Python Foundation · Lawson Dong**

A small runnable experiment about **identity**, **equality**, and **references**.

**Start here:** in Google Colab, choose **Runtime → Run all** (or run each code cell from top to bottom). Connect to a Python runtime if prompted. Then use the controls in section 3. You can edit every code cell and run your own experiments. Google sign-in may be required to run or save a copy; this public notebook is viewable without editing the original.

The screenshot shows `True`, `False`, `True`. It does not show how the variables were created. The list example below is one reliable way to reproduce those results; the outputs alone do not tell us the original variable types.

## 1. Reproduce the three results

`b = a` binds another name to the **same list**. A second list literal creates a **new list** with equal contents.

Predict the three outputs before running this cell.

In [1]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]

print(a is b)
print(c is a)
print(c == a)

True
False
True


| Expression | Question | Result in this example |
|---|---|---|
| `a is b` | Do these names refer to the same object? | `True` |
| `c is a` | Do these names refer to the same object? | `False` |
| `c == a` | Do these lists have equal contents? | `True` |

For these lists, `==` compares their contents; `is` tests object identity. Two different objects can have equal values. Assignment does **not** copy a list.

## 2. Inspect identity with `id()`

An object's ID is unique among objects that are alive at the same time, and stays the same during that object's lifetime. Exact numbers will differ between runs. IDs can be reused after an object is destroyed, so they are not permanent labels. CPython commonly uses a memory address for an ID; that is an implementation detail.

In [ ]:
for name, obj in [('a', a), ('b', b), ('c', c)]:
    print(f'{name}: value={obj!r}, id={id(obj)}')

print('a and b have the same ID:', id(a) == id(b))
print('a and c have the same ID:', id(a) == id(c))

## 3. Interactive reference lab

Run this cell, then try this sequence:

1. Leave `c = a.copy()` selected. There are two list objects but three variable names.
2. **Append to b**. Because `a` and `b` are aliases, both show the modified list. `c` stays unchanged.
3. **Match c's values to a**. Equality becomes true again without making `c` the same object.
4. **Rebind b to a copy**. This changes what `b` refers to; it does not mutate `a`.
5. Select `c = a`, then append to `b` again: all three names now refer to one list.

The diagram and comparisons come from live Python objects. The controls reset their own lab state; they do not change the `a`, `b`, and `c` variables in your other code cells. Changing the relationship selector also resets the lab.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # Also works in a local Jupyter notebook.

class ReferenceLab:
    def __init__(self):
        self.relation = widgets.Dropdown(
            options=['c = a.copy()', 'c = a', 'c = [1, 2, 3]'],
            description='Create c:', layout=widgets.Layout(width='310px'))
        self.number = widgets.IntText(value=4, description='Append:',
                                      layout=widgets.Layout(width='210px'))
        self.append_button = widgets.Button(description='Append to b', button_style='info')
        self.match_button = widgets.Button(description="Match c's values to a",
                                           layout=widgets.Layout(width='190px'))
        self.rebind_button = widgets.Button(description='Rebind b to a copy',
                                            layout=widgets.Layout(width='175px'))
        self.reset_button = widgets.Button(description='Reset')
        self.diagram = widgets.HTML()
        self.results = widgets.HTML()
        self.action = widgets.HTML()
        self.relation.observe(lambda change: self.reset(), names='value')
        self.append_button.on_click(lambda _: self.append_to_b())
        self.match_button.on_click(lambda _: self.match_c())
        self.rebind_button.on_click(lambda _: self.rebind_b())
        self.reset_button.on_click(lambda _: self.reset())
        self.ui = widgets.VBox([
            widgets.HBox([self.relation, self.number], layout=widgets.Layout(flex_flow='row wrap')),
            widgets.HBox([self.append_button, self.match_button, self.rebind_button, self.reset_button],
                         layout=widgets.Layout(flex_flow='row wrap')),
            self.action, self.diagram, self.results])
        self.reset()

    def reset(self):
        self.a = [1, 2, 3]
        self.b = self.a
        self.c = self.a if self.relation.value == 'c = a' else self.a.copy() if self.relation.value == 'c = a.copy()' else [1, 2, 3]
        self.update('Reset: a = [1, 2, 3]; b = a; ' + self.relation.value)

    def append_to_b(self):
        value = self.number.value
        self.b.append(value)
        self.update(f'b.append({value}) — mutate the list that b refers to.')

    def match_c(self):
        self.c[:] = self.a
        self.update('c[:] = a — change contents in place; keep the same c object.')

    def rebind_b(self):
        self.b = self.a.copy()
        self.update('b = a.copy() — bind b to a new list; leave a unchanged.')

    def comparisons(self):
        return [('a is b', self.a is self.b), ('c is a', self.c is self.a),
                ('c == a', self.c == self.a), ('b == a', self.b == self.a)]

    def update(self, action):
        self.action.value = '<p><code>' + html.escape(action) + '</code></p>'
        refs = [('a', self.a), ('b', self.b), ('c', self.c)]
        objects = []
        for _, obj in refs:
            if not any(obj is existing for existing in objects):
                objects.append(obj)
        colors = ['#41718c', '#a66f83', '#827ba6']
        items = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 370" role="img" aria-label="Variable names connected to their actual Python list objects" style="max-width:760px;width:100%;background:#f8f6f0;border-radius:14px">',
                 '<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0 0 L8 4 L0 8" fill="#657787"/></marker></defs>',
                 '<text x="35" y="32" font-size="13" fill="#596776">VARIABLE NAMES</text>',
                 '<text x="360" y="32" font-size="13" fill="#596776">LIVE LIST OBJECTS</text>']
        ys = [75 + i * 110 for i in range(len(objects))]
        for row, (name, obj) in enumerate(refs):
            index = next(i for i, existing in enumerate(objects) if obj is existing)
            y = 92 + row * 110
            items.extend([f'<rect x="35" y="{y-25}" width="95" height="50" rx="10" fill="white" stroke="{colors[index]}"/>',
                          f'<text x="82" y="{y+7}" text-anchor="middle" font-size="24" fill="{colors[index]}">{name}</text>',
                          f'<path d="M135 {y} C245 {y},245 {ys[index]+25},345 {ys[index]+25}" fill="none" stroke="{colors[index]}" stroke-width="2.5" marker-end="url(#arrow)"/>'])
        for i, obj in enumerate(objects):
            y = ys[i]
            value = html.escape(repr(obj))
            if len(value) > 48:
                value = html.escape(repr(obj)[:43] + '…]')
            items.extend([f'<rect x="355" y="{y-16}" width="380" height="86" rx="12" fill="white" stroke="{colors[i]}"/>',
                          f'<text x="375" y="{y+8}" font-size="13" fill="{colors[i]}">Object {i+1} · id={id(obj)}</text>',
                          f'<text x="375" y="{y+42}" font-size="18" font-family="monospace" fill="#273845">{value}</text>'])
        items.append('</svg>')
        self.diagram.value = ''.join(items)
        rows = ''.join(f'<tr><td style="padding:8px 22px"><code>{expr}</code></td><td style="padding:8px;color:{"#28705d" if result else "#a34f60"}"><b>{result}</b></td></tr>' for expr, result in self.comparisons())
        self.results.value = '<table><tr><th>Expression</th><th>Actual result</th></tr>' + rows + '</table>'

lab = ReferenceLab()
display(lab.ui)

## 4. Mutation vs rebinding: edit and run

This uses ordinary Python, without widgets. Change the values or add more operations. Predict which lists change before running it.

In [ ]:
a = [1, 2, 3]
b = a
c = a.copy()

b.append(4)
print('After b.append(4):', 'a =', a, 'b =', b, 'c =', c)

b = [1, 2, 3]  # Rebind the name b; do not modify the old list.
print('After rebinding b:', 'a =', a, 'b =', b, 'c =', c)
print('b is c:', b is c)
print('b == c:', b == c)

## 5. A shallow copy still shares nested objects

`list.copy()` creates a new **outer list**. It does not recursively copy the objects inside it. Run this example to see the difference.

In [ ]:
outer = [[1, 2], [3, 4]]
shallow = outer.copy()
print('Same outer list?', outer is shallow)
print('Same first inner list?', outer[0] is shallow[0])
shallow[0].append(99)
print('outer:', outer)
print('shallow:', shallow)

from copy import deepcopy
deep = deepcopy(outer)
print('Deep copy shares first inner list?', outer[0] is deep[0])

## 6. What to use in real code

- Use `==` for equality of list contents, strings, or numeric values.
- Use `is` when you deliberately want identity, commonly `value is None`.
- Do not infer a general rule from `is` on numbers or strings: Python implementations may reuse immutable objects. The list examples above avoid that ambiguity.
- Equality is type-dependent and can be customized by `__eq__`; it is not universally a byte-for-byte comparison.

Try this last cell after the interactive lab. It checks the original three statements using fresh lists, independently of the controls.

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
assert a is b
assert c is not a
assert c == a
print('All three predictions are correct.')

## References

[Python: identity comparisons](https://docs.python.org/3/reference/expressions.html#identity-comparisons) · [Python: objects, values and types](https://docs.python.org/3/reference/datamodel.html#objects-values-and-types) · [Python: id()](https://docs.python.org/3/library/functions.html#id) · [Colab: interactive forms and widgets](https://colab.research.google.com/notebooks/forms.ipynb)